## L1-Penalized GP with Bilby

*Notes*:

- In BL, using 10,000 iterations with 1000 iterations of burn-in. 

- Gamma prior on lambda squared with r = 1, delta = 1.78


*To Do*:
- run sanity checks
    - take posterior likelihood from BL, check values with likelihood compared to random values
- test different samplers
    - 0.2 < acceptance fraction < 0.5
    - look at trace plots and how many samples are needed for convergence
    - try: pyemcee

- two pieces of this analysis
    - can this method reproduce some of the main results?
    - also some of a way of getting some uncertainty analysis


- he is sending guide to running jupyter notebook on a quest server






#### Another idea for beta
- Having issues with sampling and then filling in the priors
- Can i calculate beta as a deterministic value using another random variable that i sample? 

https://arxiv.org/pdf/1312.0906 - non-centered parameterizations? page 3

In this paper, they have $y_i \sim \mathcal(\theta_i, \sigma_i^2)$ where $\theta_i \sim \mathcal(\mu, \tau^2)$. They then use $\theta_i = \mu + \tau*\nu_i$, where $\nu_i \sim \mathcal{N}(0,1)$.


So in our case, we have $ \beta \mid \sigma^2_N, \tau^2$, where $\beta \sim \mathcal{N}(0, \sigma_N^2D_T)$. Could we use $ \beta \mid \sigma^2_N, \tau^2, z $, where $ z \sim \mathcal{N}(0, I)$?

then $ \beta = 0 + \sqrt{\sigma_N^2D_T} * z$?

### Hidden

\begin{align*}
& p(y, \beta, \tau^2, \sigma^2_N, \sigma^2_{GP}, \ell, \lambda) \\
&= \frac{\exp\!\left(
 -\frac12 (y - X\beta)^\top (K + \sigma^2_N I_n)^{-1} (y - X\beta)
\right)}
{\sqrt{(2\pi)^n |K + \sigma^2_N I_n|}} \\[6pt]
&\quad \times \frac{\exp\!\left(-\frac12 \beta^\top (\sigma^2_N D_\tau)^{-1} \beta \right)}
{(2\pi)^{p/2} |\sigma^2_N D_\tau|^{1/2}} \\[6pt]
&\quad \times \prod_{j=1}^p \frac{\lambda^2}{2} \exp\!\left( -\frac{\lambda^2}{2} \tau_j^2 \right) \\[6pt]
&\quad \times \frac{\beta_{\text{noise}}^{\alpha_{\text{noise}}}}{\Gamma(\alpha_{\text{noise}})}
(\sigma^2_N)^{-\alpha_{\text{noise}}-1} \exp\!\left( -\frac{\beta_{\text{noise}}}{\sigma^2_N} \right) \\[6pt]
&\quad \times \frac{\beta_{\text{GP}}^{\alpha_{\text{GP}}}}{\Gamma(\alpha_{\text{GP}})}
(\sigma^2_{GP})^{-\alpha_{\text{GP}}-1} \exp\!\left( -\frac{\beta_{\text{GP}}}{\sigma^2_{GP}} \right) \\[6pt]
&\quad \times \frac{1}{\ell \, \sigma_\ell \sqrt{2\pi}}
\exp\!\left( -\frac{(\log \ell - \mu_\ell)^2}{2\sigma_\ell^2} \right) \\[6pt]
&\quad \times \frac{b_\lambda^{a_\lambda}}{\Gamma(a_\lambda)} \lambda^{a_\lambda - 1} e^{-b_\lambda \lambda}
\end{align*}

Posterior:
$$p(\beta, \tau^2, \sigma^2_N, \sigma^2_{GP}, \ell, \lambda \mid y) \propto p(y | \beta, \sigma^2_N, \sigma^2_{GP}, \ell) \times p(\beta | \sigma^2_N, \tau^2) \times \prod_{j=1}^p p(\tau_j^2 | \lambda) \times p(\sigma^2_N) \times p(\sigma^2_{GP}) \times p(\ell) \times p(\lambda)$$

In [62]:
## to save time plotting


# fig = plt.Figure() # notice the capital F
# sns.pairplot(.....)
# plt.savefig(.....)
# plt.close()


# pairplot(corner = True) # only plots half of symmetrical plot


### Data

In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import bilby
from bilby.core.utils import random
import json
import scipy.special
import scipy.stats as stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


# set up
random.seed(123)

# label = "penalizedGP"
# outdir = "outdir2"

label = "emcee_small"
outdir = "sampler_testing"

bilby.utils.check_directory_exists_and_if_not_mkdir(outdir)

In [64]:
### --- to load diabetes data ---
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True)
X = diabetes.data

selected_features = ['bmi', 'bp', 's1']
X = X[selected_features]

label_names = diabetes.feature_names
y = diabetes.target

Xtrain, Xtest, ytrain, ytest = train_test_split(X, y, test_size=0.2, random_state=22)

scaler = StandardScaler()
Xtrain = scaler.fit_transform(Xtrain) # fit and scale training data
Xtest = scaler.transform(Xtest) # scale test data

### MCMC

In [65]:
# helper function to compute the RBF kernel
def rbf_kernel(X, ell, sigma_gp):
    N = X.shape[0]
    K = np.zeros((N, N))
    for i in range(N):
        for j in range(N):
            diff = (X[i] - X[j]) / ell
            K[i, j] = sigma_gp**2 * np.exp(-0.5 * np.dot(diff, diff))
    return K

In [66]:
# custom likelihood for penalized GP regression

class PenalizedGPLikelihood(bilby.Likelihood):
    def __init__(self, X, y):
        # store data
        self.X = np.asarray(X)
        self.y = np.asarray(y)

        # define parameters
        parameters = {}

        # linear coefficients (betas)
        for i in range(self.X.shape[1]):
            parameters[f"beta{i}"] = None

        # variance parameters (tau^2)
        for i in range(self.X.shape[1]):
            parameters[f"tau_sq_{i}"] = None

        # noise parameters
        parameters["inv_sigma_noise"] = None # remember this is gamma not inverse gamma, so use (alpha, 1/b)
        parameters["inv_sigma_gp"] = None # remember this is gamma not inverse gamma, so use (alpha, 1/b)
        
        # lengthscales (ells)
        for i in range(self.X.shape[1]):
            parameters[f"ell{i}"] = None

        # lambda (for L1 regularization)
        parameters["lambda2"] = None
        
        super().__init__(parameters=parameters)


    def log_likelihood(self):

        # extract parameters
        betas = np.array([self.parameters[f"beta{i}"] for i in range(self.X.shape[1])])
        tau_sqs = np.array([self.parameters[f"tau_sq_{i}"] for i in range(self.X.shape[1])])
        inv_sigma_noise = self.parameters["inv_sigma_noise"]
        inv_sigma_gp = self.parameters["inv_sigma_gp"]
        ells = np.array([self.parameters[f"ell{i}"] for i in range(self.X.shape[1])])
        lbda2 = self.parameters["lambda2"] # sample lambda squared
        lbda = np.sqrt(lbda2) # take square root to get lambda
        
        # calculate n and p
        n = self.X.shape[0]
        p = self.X.shape[1]

        # converting between sampled gamma to desired inverse gamma
        # check over this
        sigma_gp = 1/inv_sigma_gp if inv_sigma_gp is not None else None
        sigma_noise = 1/inv_sigma_noise if inv_sigma_noise is not None else None

        # calculate covariance matrix C
        C = rbf_kernel(self.X, ells, sigma_gp)

        # sample beta values here using tau_sqs, sigma_noise
        names = [f"beta{i}" for i in range(self.X.shape[1])] # names for multivariate gaussian
        mu = np.zeros(p) # mean vector for betas
        cov = (sigma_noise**2) * np.diag(tau_sqs) # covariance matrix for betas

        # beta_sample = bilby.core.prior.MultivariateGaussian(names, mu, cov).sample() # 
        # beta_arr = np.array([beta_sample[f"beta{i}"] for i in range(self.X.shape[1])])
        # beta_arr = np.random.multivariate_normal(mu, cov) # what if we directly sample from multivariate normal

        # calculate residuals
        residuals = self.y - self.X @ betas

        # calculate diagonal matrix D
        D = np.diag(tau_sqs)

        # --- log likelihood components ---
        log_lik_gp = (
            -0.5 * n * np.log(2 * np.pi) 
            -0.5 * np.linalg.slogdet(C + sigma_noise**2 * np.eye(n))[1]
            -0.5 *(residuals).T @ np.linalg.solve(C + sigma_noise**2 * np.eye(n), residuals))

        log_lik_beta = (
            -0.5 * p * np.log(2 * np.pi)
            -0.5 * np.linalg.slogdet(sigma_noise**2 * D)[1]
            -0.5 * betas.T @ np.linalg.solve(sigma_noise**2 * D, betas)
        )
        log_likelihood = (log_lik_gp + log_lik_beta)

        return log_likelihood


In [67]:
# make priors
priors = dict()

# --- from testing of sampling beta in likelihood ---
# sample initial beta values from a wide uniform prior 
# if 'beta_sample' not in locals(): 
#     beta_sample = {}
#     for i in range(Xtrain.shape[1]):
#         beta_sample[f"beta{i}"] = bilby.core.prior.Uniform(-10, 10, name=f"beta{i}") # wide

for i in range(Xtrain.shape[1]):
    # priors[f"beta{i}"] = beta_sample[f"beta{i}"]  # reassign sampled values back to priors here
    priors[f"beta{i}"] = bilby.core.prior.Uniform(-10, 10, name=f"beta{i}") # use for testing

    priors[f"tau_sq_{i}"] = bilby.core.prior.Exponential(0.5, f"tau_sq_{i}")
    priors[f"ell{i}"] = bilby.core.prior.LogNormal(0, 1, f"ell{i}") # define log-normal priors for each lengthscale

priors["inv_sigma_noise"] = bilby.core.prior.Gamma(1, 1, "inv_sigma_noise")  
priors["inv_sigma_gp"] = bilby.core.prior.Gamma(1, 1, "inv_sigma_gp")  

priors["lambda2"] = bilby.core.prior.Gamma(1.0, 1.78, name="lambda2") # positive

# define the likelihood function that we defined earlier
likelihood = PenalizedGPLikelihood(
    X = Xtrain,
    y = ytrain)

In [68]:
print(type(priors))
print(priors)

<class 'dict'>
{'beta0': Uniform(minimum=-10, maximum=10, name='beta0', latex_label='beta0', unit=None, boundary=None), 'tau_sq_0': Exponential(mu=0.5, name='tau_sq_0', latex_label='tau_sq_0', unit=None, boundary=None), 'ell0': LogNormal(mu=0, sigma=1, name='ell0', latex_label='ell0', unit=None, boundary=None), 'beta1': Uniform(minimum=-10, maximum=10, name='beta1', latex_label='beta1', unit=None, boundary=None), 'tau_sq_1': Exponential(mu=0.5, name='tau_sq_1', latex_label='tau_sq_1', unit=None, boundary=None), 'ell1': LogNormal(mu=0, sigma=1, name='ell1', latex_label='ell1', unit=None, boundary=None), 'beta2': Uniform(minimum=-10, maximum=10, name='beta2', latex_label='beta2', unit=None, boundary=None), 'tau_sq_2': Exponential(mu=0.5, name='tau_sq_2', latex_label='tau_sq_2', unit=None, boundary=None), 'ell2': LogNormal(mu=0, sigma=1, name='ell2', latex_label='ell2', unit=None, boundary=None), 'inv_sigma_noise': Gamma(k=1, theta=1, name='inv_sigma_noise', latex_label='inv_sigma_noise',

In [69]:
# run MCMC sampler
result = bilby.run_sampler(
    likelihood=likelihood, # likelihood function
    priors=priors, # prior distributions
    sampler="emcee", # other options for mcmc are emcee, zeus, ptemcee, bilby-mcmc 
    nwalkers = 200,
    nsteps = 50,
    nburn = 20,
    outdir=outdir,
    label=label)

11:02 bilby INFO    : Running for label 'emcee_small', output will be saved to 'sampler_testing'
11:02 bilby INFO    : Analysis priors:
11:02 bilby INFO    : beta0=Uniform(minimum=-10, maximum=10, name='beta0', latex_label='beta0', unit=None, boundary=None)
11:02 bilby INFO    : tau_sq_0=Exponential(mu=0.5, name='tau_sq_0', latex_label='tau_sq_0', unit=None, boundary=None)
11:02 bilby INFO    : ell0=LogNormal(mu=0, sigma=1, name='ell0', latex_label='ell0', unit=None, boundary=None)
11:02 bilby INFO    : beta1=Uniform(minimum=-10, maximum=10, name='beta1', latex_label='beta1', unit=None, boundary=None)
11:02 bilby INFO    : tau_sq_1=Exponential(mu=0.5, name='tau_sq_1', latex_label='tau_sq_1', unit=None, boundary=None)
11:02 bilby INFO    : ell1=LogNormal(mu=0, sigma=1, name='ell1', latex_label='ell1', unit=None, boundary=None)
11:02 bilby INFO    : beta2=Uniform(minimum=-10, maximum=10, name='beta2', latex_label='beta2', unit=None, boundary=None)
11:02 bilby INFO    : tau_sq_2=Exponenti

### Analysis of Results

In [90]:
emcee_small_results = bilby.result.read_in_result(f'sampler_testing/emcee_small_result.json')   # or result.h5
emcee_small_results.plot_walkers()

ImportError: cannot import name 'LoadFlags' from 'matplotlib.ft2font' (/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/matplotlib/ft2font.cpython-310-darwin.so)

In [31]:
beta_samples = np.array([result.posterior[f"beta{i}"] for i in range(X.shape[1])]).T

beta_medians = np.median(beta_samples, axis=0)
lower = np.percentile(beta_samples, 2.5, axis=0)
upper = np.percentile(beta_samples, 97.5, axis=0)

table = pd.DataFrame({
    "Variable": [label_names[i] for i in range(beta_samples.shape[1])],
    "Posterior Median": beta_medians,
    "95% Credible Interval": [f"({l:.2f}, {u:.2f})" for l, u in zip(lower, upper)]})

In [32]:
table

,Variable,Posterior Median,95% Credible Interval
0,age,35.547817,"(-83.46, 174.12)"
1,sex,-20.431143,"(-214.76, 108.46)"
2,bmi,-5.288393,"(-120.67, 101.08)"


In [26]:
ell_samples = np.array([result.posterior[f"ell{i}"] for i in range(X.shape[1])]).T
ell_medians = np.median(ell_samples, axis=0)
ell_lower = np.percentile(ell_samples, 2.5, axis=0)
ell_upper = np.percentile(ell_samples, 97.5, axis=0)
ell_table = pd.DataFrame({
    "Variable": [f"ell_{i+1}" for i in range(ell_samples.shape[1])],
    "Posterior Median": ell_medians,
    "95% Credible Interval": [f"({l:.2f}, {u:.2f})" for l, u in zip(ell_lower, ell_upper)]})

In [27]:
ell_table

,Variable,Posterior Median,95% Credible Interval
0,ell_1,1.590548,"(0.92, 2.72)"
1,ell_2,1.677875,"(0.93, 2.99)"
2,ell_3,1.551204,"(0.64, 2.77)"


In [ ]:
beta_medians = np.median(beta_samples, axis=0)
ell_medians = np.median(ell_samples, axis=0)
sigma_gp_median = ...
sigma_noise_median = ...



### Creating GP with Sampled Parameters

In [29]:
# scale data back
Xtrain_orig = scaler.inverse_transform(Xtrain)
Xtest_orig = scaler.inverse_transform(Xtest)


# create gaussian p
import gpflow

# convert data to float64 for gpflow
Xtrain_gp = Xtrain.astype(np.float64)
ytrain_gp = ytrain.reshape(-1, 1).astype(np.float64)
Xtest_gp = Xtest.astype(np.float64)
ytest_gp = ytest.reshape(-1, 1).astype(np.float64)

# define kernel and model
kernel = gpflow.kernels.SquaredExponential(lengthscales=ell_medians, variance = sigma_gp_median**2) # set kernel with sampled lengthscales and variance

mean_fn = gpflow.mean_functions.Linear(A=np.diag(beta_medians), b=0.0) # set linear mean function

model = gpflow.models.GPR(data=(Xtrain_gp, ytrain_gp), kernel=kernel, mean_function=meanfn)

# use sampled parameters
model.parameters['lengthscales'].assign(ell_medians) # ???

